# Sandbox Analysis: Natural Language Bioeconomy Exploration

Welcome to the AI-driven data sandbox for the `ca-biositing` project. This notebook uses **PandasAI 3.0+** and the **CBORG API** (Gemini models) to allow you to explore geospatial bioeconomy data using natural language.

### Capabilities:
- **Natural Language Querying**: Ask questions about the data in plain English.
- **Interactive Visualizations**: Generate Plotly charts for zooming and hovering.
- **Local DB Integration**: Queries the live `ca_biositing` PostgreSQL instance.

## 1. Setup and Environment

We initialize our connection to the CBORG LLM gateway and apply a **Global Patch** to PandasAI to ensure all interactive charts render directly in the notebook.

In [ ]:
import sys
import os
from pathlib import Path

print(f"Python Executable: {sys.executable}")
print(f"Current Working Directory: {os.getcwd()}")

# Force discovery of local modules in analysis/ai_exploration
def find_setup_dir():
    # Try relative to root, then relative to current
    for p in [Path("analysis/ai_exploration"), Path(".")]:
        if (p / "sandbox_setup.py").exists():
            return p.resolve()
    return None

setup_dir = find_setup_dir()
if setup_dir:
    if str(setup_dir) not in sys.path:
        sys.path.insert(0, str(setup_dir))
        print(f"Added {setup_dir} to sys.path")
else:
    print("WARNING: Could not find sandbox_setup.py directory automatically.")

try:
    from sandbox_setup import init_sandbox, get_agent_with_metadata
    print("SUCCESS: sandbox_setup imported")
except ImportError as e:
    print(f"FAILURE: {e}")
    print(f"Path: {sys.path}")
    raise

llm, engine = init_sandbox(model_name="gemini-2.0-flash")

## 3. Loading Data and Initializing Agent

We load all relevant views into the agent.

In [ ]:
views = ["analysis_data_view", "usda_census_view", "analysis_average_view"]

# Initialize the Agent with SQL Connectors and schema-aware metadata
agent = get_agent_with_metadata(llm, engine, views)

print(f"\nAgent initialized with {len(views)} views and custom SQL skill.")

## 4. Interactive Queries

Ask for an interactive chart below. The global patch ensures it renders as interactive HTML in the cell output.

In [ ]:
agent.chat("What are the top 5 most common resources in the analysis_data_view?")

### A. Interactive Visualization
Visualize the energy content distribution with an interactive Plotly bar chart.

In [ ]:
agent.chat("Can you also provide the unit with the last chart you generated?")

In [ ]:
agent.chat("Can you give me the python code you used to generate the previous plot?")

In [ ]:
agent.last_code_executed

### B. Scatter Plot
Visualize relationship between two variables from the average view.

In [ ]:
agent.chat("Plot xylose vs glucose for all resources, averaged across all geoid in analysis_average_view. " \
            "Make the plot an interactive plotly plot with the tooltip showing resource name.")

In [ ]:
agent.chat("Using the analysis_average_view, can you make me a summary of all analysis, grouped by analysis types into different charts, for just almond_shells. I want this averaged across geoid, and faceted bar plots with all parameters for each analysis type present in the data. Basically this will be a multi paneled plot, displayed in the viewer")

### C. Multi-DataFrame Analysis
Correlate production data with USDA census data.

In [ ]:
agent.chat("Summarize how the county-level production in the analysis data relates to the census data in the USDA view.")